In [1]:
!pip install yfinance pandas numpy scipy matplotlib seaborn statsmodels

In [2]:
import os
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)
print("folders created")

folders created


In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

START_DATE = "2010-01-01"
END_DATE   = "2025-12-31"

TICKERS = [
    "AAPL", "MSFT", "NVDA", "GOOGL", "META", "AMZN", "TSM", "AVGO", "ORCL", "ASML",
    "JPM", "BAC", "WFC", "GS", "MS", "BLK", "AXP", "SPGI", "MCO", "ICE",
    "JNJ", "UNH", "LLY", "PFE", "ABBV", "MRK", "TMO", "ABT", "DHR", "BMY",
    "PG", "KO", "PEP", "WMT", "COST", "MCD", "NKE", "SBUX", "TGT", "HD",
    "XOM", "CVX", "CAT", "DE", "HON", "UPS", "BA", "MMM", "GE", "LMT"
]

print("downloading prices...")
prices = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)["Close"]
print(f"got {prices.shape[0]} days, {prices.shape[1]} stocks")

print("downloading SPY...")
spy = yf.download("SPY", start=START_DATE, end=END_DATE, auto_adjust=True)["Close"]
spy_returns = spy.pct_change().dropna()

print("computing returns...")
returns = prices.pct_change().dropna()

missing_pct  = returns.isnull().sum() / len(returns)
good_tickers = missing_pct[missing_pct < 0.20].index
returns      = returns[good_tickers].fillna(0)
prices       = prices[good_tickers]

common_dates = returns.index.intersection(spy_returns.index)
returns      = returns.loc[common_dates]
spy_returns  = spy_returns.loc[common_dates]
prices       = prices.loc[common_dates]

print(f"{len(common_dates)} trading days")
print(f"from {common_dates[0].date()} to {common_dates[-1].date()}")
print(f"{len(good_tickers)} stocks after cleaning")

returns.to_csv("data/stock_returns.csv")
spy_returns.to_csv("data/spy_returns.csv")
prices.to_csv("data/stock_prices.csv")

print("done")

downloading prices...


[*********************100%***********************]  50 of 50 completed


got 4023 days, 50 stocks
downloading SPY...


[*********************100%***********************]  1 of 1 completed


computing returns...
3268 trading days
from 2013-01-03 to 2025-12-30
50 stocks after cleaning
done


In [4]:
import pandas as pd
import numpy as np

print("loading data...")
returns = pd.read_csv("data/stock_returns.csv", index_col=0, parse_dates=True)
prices  = pd.read_csv("data/stock_prices.csv",  index_col=0, parse_dates=True)

print("building factors...")

# momentum - 12 month return skipping last month
momentum = (
    (1 + returns).rolling(252).apply(np.prod, raw=True) /
    (1 + returns).rolling(21).apply(np.prod, raw=True)
) - 1

# value - distance from 52 week high
high_52w = prices.rolling(252).max()
value    = 1 - (prices / high_52w)
value    = value.loc[returns.index]

# quality - rolling sharpe ratio
quality = returns.rolling(63).mean() / (returns.rolling(63).std() + 1e-8)

# size - price rank proxy
size = prices.rank(axis=1, ascending=True).loc[returns.index]

# low vol - inverse rolling volatility
low_vol = 1 / (returns.rolling(63).std() + 1e-8)

# profitability - 1 year cumulative return
profitability = (1 + returns).rolling(252).apply(np.prod, raw=True) - 1

# drop warmup period
warmup = 252
factors = {
    "momentum":      momentum.iloc[warmup:],
    "value":         value.iloc[warmup:],
    "quality":       quality.iloc[warmup:],
    "size":          size.iloc[warmup:],
    "low_vol":       low_vol.iloc[warmup:],
    "profitability": profitability.iloc[warmup:]
}
returns_cut = returns.iloc[warmup:]

print(f"{returns_cut.shape[0]} days after warmup")

# save
for name, df in factors.items():
    df.to_csv(f"data/factor_{name}.csv")
returns_cut.to_csv("data/returns_cut.csv")

print("done")

loading data...
building factors...
3016 days after warmup
done


In [5]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

print("loading data...")
returns = pd.read_csv("data/returns_cut.csv", index_col=0, parse_dates=True)
spy     = pd.read_csv("data/spy_returns.csv", index_col=0, parse_dates=True).squeeze()

factor_names = ["momentum", "value", "quality", "size", "low_vol", "profitability"]
factors = {}
for name in factor_names:
    factors[name] = pd.read_csv(f"data/factor_{name}.csv", index_col=0, parse_dates=True)

# align dates
common = returns.index
for name in factor_names:
    common = common.intersection(factors[name].index)
common = common.intersection(spy.index)

returns = returns.loc[common]
spy     = spy.loc[common]
for name in factor_names:
    factors[name] = factors[name].loc[common]

print(f"{len(common)} days")

# long short portfolio for each factor
# buy top 20% of stocks, short bottom 20%
def long_short_portfolio(factor_df, returns_df, top_pct=0.2, bottom_pct=0.2):
    port_returns = []
    dates        = []
    month_ends   = factor_df.resample("ME").last().index
    for i in range(len(month_ends) - 1):
        rd = month_ends[i]
        nd = month_ends[i + 1]
        if rd not in factor_df.index:
            continue
        scores     = factor_df.loc[rd].dropna()
        n          = len(scores)
        top_n      = max(1, int(n * top_pct))
        bottom_n   = max(1, int(n * bottom_pct))
        long_stocks  = scores.nlargest(top_n).index.tolist()
        short_stocks = scores.nsmallest(bottom_n).index.tolist()
        mask   = (returns_df.index > rd) & (returns_df.index <= nd)
        period = returns_df.loc[mask]
        if period.empty:
            continue
        long_ret  = period[long_stocks].mean(axis=1)
        short_ret = period[short_stocks].mean(axis=1)
        ls_ret    = long_ret - short_ret
        for date, ret in ls_ret.items():
            port_returns.append(ret)
            dates.append(date)
    return pd.Series(port_returns, index=dates).sort_index()

print("computing long/short portfolios...")
ls_returns = {}
for name in factor_names:
    ls_returns[name] = long_short_portfolio(factors[name], returns)
    print(f"  {name} done")

# performance metrics
def metrics(r, label):
    ann_ret = r.mean() * 252
    ann_vol = r.std()  * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0
    cum     = (1 + r).cumprod()
    max_dd  = ((cum - cum.cummax()) / cum.cummax()).min()
    return {
        "factor":          label,
        "ann. return":     f"{ann_ret:.2%}",
        "ann. volatility": f"{ann_vol:.2%}",
        "sharpe":          f"{sharpe:.2f}",
        "max drawdown":    f"{max_dd:.2%}"
    }

print("\nlong/short factor performance:")
results = []
for name in factor_names:
    m = metrics(ls_returns[name], name)
    results.append(m)
    print(f"  {name}: return={m['ann. return']}, sharpe={m['sharpe']}")

pd.DataFrame(results).to_csv("results/factor_performance.csv", index=False)

# chart 1 - cumulative returns for each factor
print("\ngenerating charts...")
fig, ax = plt.subplots(figsize=(14, 7))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
for i, name in enumerate(factor_names):
    cum = (1 + ls_returns[name]).cumprod()
    ax.plot(cum.index, cum.values, label=name, color=colors[i], linewidth=1.2)
ax.set_title("long/short factor cumulative returns")
ax.set_ylabel("cumulative return")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/factor_cumulative_returns.png", dpi=150, bbox_inches="tight")
plt.close()
print("saved factor_cumulative_returns.png")

# chart 2 - rolling 252 day sharpe for each factor
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()
for i, name in enumerate(factor_names):
    r = ls_returns[name]
    rolling_sharpe = (r.rolling(252).mean() * 252) / (r.rolling(252).std() * np.sqrt(252))
    axes[i].plot(rolling_sharpe.index, rolling_sharpe.values, color=colors[i], linewidth=1)
    axes[i].axhline(0, color="black", linewidth=0.5, linestyle="--")
    axes[i].set_title(f"{name} rolling sharpe")
    axes[i].grid(True, alpha=0.3)
plt.suptitle("rolling 1-year sharpe ratio by factor", fontsize=13)
plt.tight_layout()
plt.savefig("results/rolling_sharpe.png", dpi=150, bbox_inches="tight")
plt.close()
print("saved rolling_sharpe.png")

# chart 3 - factor correlation matrix
ls_df    = pd.DataFrame(ls_returns)
corr     = ls_df.corr()
fig, ax  = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            ax=ax, square=True, linewidths=0.5)
ax.set_title("factor return correlation matrix")
plt.tight_layout()
plt.savefig("results/factor_correlation.png", dpi=150, bbox_inches="tight")
plt.close()
print("saved factor_correlation.png")

print("\ndone")

loading data...
3016 days
computing long/short portfolios...
  momentum done
  value done
  quality done
  size done
  low_vol done
  profitability done

long/short factor performance:
  momentum: return=7.65%, sharpe=0.40
  value: return=-4.70%, sharpe=-0.24
  quality: return=6.47%, sharpe=0.37
  size: return=-5.13%, sharpe=-0.46
  low_vol: return=-4.60%, sharpe=-0.22
  profitability: return=10.65%, sharpe=0.53

generating charts...
saved factor_cumulative_returns.png
saved rolling_sharpe.png
saved factor_correlation.png

done


In [6]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("loading long/short returns...")
returns  = pd.read_csv("data/returns_cut.csv", index_col=0, parse_dates=True)
spy      = pd.read_csv("data/spy_returns.csv", index_col=0, parse_dates=True).squeeze()

factor_names = ["momentum", "value", "quality", "size", "low_vol", "profitability"]
factors = {}
for name in factor_names:
    factors[name] = pd.read_csv(f"data/factor_{name}.csv", index_col=0, parse_dates=True)

common = returns.index
for name in factor_names:
    common = common.intersection(factors[name].index)
common = common.intersection(spy.index)

returns = returns.loc[common]
spy     = spy.loc[common]
for name in factor_names:
    factors[name] = factors[name].loc[common]

def long_short_portfolio(factor_df, returns_df, top_pct=0.2, bottom_pct=0.2):
    port_returns = []
    dates        = []
    month_ends   = factor_df.resample("ME").last().index
    for i in range(len(month_ends) - 1):
        rd = month_ends[i]
        nd = month_ends[i + 1]
        if rd not in factor_df.index:
            continue
        scores       = factor_df.loc[rd].dropna()
        n            = len(scores)
        top_n        = max(1, int(n * top_pct))
        bottom_n     = max(1, int(n * bottom_pct))
        long_stocks  = scores.nlargest(top_n).index.tolist()
        short_stocks = scores.nsmallest(bottom_n).index.tolist()
        mask         = (returns_df.index > rd) & (returns_df.index <= nd)
        period       = returns_df.loc[mask]
        if period.empty:
            continue
        ls_ret = period[long_stocks].mean(axis=1) - period[short_stocks].mean(axis=1)
        for date, ret in ls_ret.items():
            port_returns.append(ret)
            dates.append(date)
    return pd.Series(port_returns, index=dates).sort_index()

ls_returns = {}
for name in factor_names:
    ls_returns[name] = long_short_portfolio(factors[name], returns)

# t-test for each factor
# tests whether the mean return is significantly different from zero
# t-stat > 2 and p-value < 0.05 means the result is statistically significant
# i.e. unlikely to be due to random chance
print("\nsignificance tests (t-test vs zero):")
print(f"{'factor':<15} {'mean return':>12} {'t-stat':>8} {'p-value':>10} {'significant':>12}")
print("-" * 60)

sig_results = []
for name in factor_names:
    r        = ls_returns[name].dropna()
    t_stat, p_value = stats.ttest_1samp(r, 0)
    ann_ret  = r.mean() * 252
    sig      = "YES" if p_value < 0.05 else "NO"
    print(f"{name:<15} {ann_ret:>12.2%} {t_stat:>8.2f} {p_value:>10.4f} {sig:>12}")
    sig_results.append({
        "factor":       name,
        "ann. return":  f"{ann_ret:.2%}",
        "t-stat":       round(t_stat, 2),
        "p-value":      round(p_value, 4),
        "significant":  sig
    })

pd.DataFrame(sig_results).to_csv("results/significance_tests.csv", index=False)

# multiple testing correction - bonferroni
# when you test 6 factors at once, you expect some to look significant by chance
# bonferroni correction makes the threshold stricter
bonferroni_threshold = 0.05 / len(factor_names)
print(f"\nbonferroni corrected threshold: {bonferroni_threshold:.4f}")
print("factors significant after correction:")
for r in sig_results:
    if float(r["p-value"]) < bonferroni_threshold:
        print(f"  {r['factor']}: p={r['p-value']}")
    else:
        print(f"  {r['factor']}: NOT significant after correction")

# chart - t-statistics
fig, ax = plt.subplots(figsize=(10, 5))
t_stats = [r["t-stat"] for r in sig_results]
colors  = ["#2ca02c" if t > 2 else "#d62728" if t < -2 else "#ff7f0e" for t in t_stats]
ax.bar(factor_names, t_stats, color=colors, alpha=0.7)
ax.axhline(2,  color="green", linestyle="--", linewidth=1, label="p=0.05 threshold")
ax.axhline(-2, color="green", linestyle="--", linewidth=1)
ax.axhline(0,  color="black", linewidth=0.5)
ax.set_title("factor t-statistics (green bars = statistically significant)")
ax.set_ylabel("t-statistic")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/significance_tests.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nsaved significance_tests.png")
print("done")

loading long/short returns...

significance tests (t-test vs zero):
factor           mean return   t-stat    p-value  significant
------------------------------------------------------------
momentum               7.65%     1.15     0.2521           NO
value                 -4.70%    -0.68     0.4936           NO
quality                6.47%     1.06     0.2883           NO
size                  -5.13%    -1.33     0.1840           NO
low_vol               -4.60%    -0.65     0.5179           NO
profitability         10.65%     1.52     0.1280           NO

bonferroni corrected threshold: 0.0083
factors significant after correction:
  momentum: NOT significant after correction
  value: NOT significant after correction
  quality: NOT significant after correction
  size: NOT significant after correction
  low_vol: NOT significant after correction
  profitability: NOT significant after correction

saved significance_tests.png
done


In [7]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("loading data...")
returns = pd.read_csv("data/returns_cut.csv", index_col=0, parse_dates=True)
spy     = pd.read_csv("data/spy_returns.csv", index_col=0, parse_dates=True).squeeze()

factor_names = ["momentum", "value", "quality", "size", "low_vol", "profitability"]
factors = {}
for name in factor_names:
    factors[name] = pd.read_csv(f"data/factor_{name}.csv", index_col=0, parse_dates=True)

common = returns.index
for name in factor_names:
    common = common.intersection(factors[name].index)
common = common.intersection(spy.index)

returns = returns.loc[common]
spy     = spy.loc[common]
for name in factor_names:
    factors[name] = factors[name].loc[common]

def long_short_portfolio(factor_df, returns_df, top_pct=0.2, bottom_pct=0.2):
    port_returns = []
    dates        = []
    month_ends   = factor_df.resample("ME").last().index
    for i in range(len(month_ends) - 1):
        rd = month_ends[i]
        nd = month_ends[i + 1]
        if rd not in factor_df.index:
            continue
        scores       = factor_df.loc[rd].dropna()
        n            = len(scores)
        top_n        = max(1, int(n * top_pct))
        bottom_n     = max(1, int(n * bottom_pct))
        long_stocks  = scores.nlargest(top_n).index.tolist()
        short_stocks = scores.nsmallest(bottom_n).index.tolist()
        mask         = (returns_df.index > rd) & (returns_df.index <= nd)
        period       = returns_df.loc[mask]
        if period.empty:
            continue
        ls_ret = period[long_stocks].mean(axis=1) - period[short_stocks].mean(axis=1)
        for date, ret in ls_ret.items():
            port_returns.append(ret)
            dates.append(date)
    return pd.Series(port_returns, index=dates).sort_index()

def sharpe(r):
    return (r.mean() * 252) / (r.std() * np.sqrt(252)) if r.std() > 0 else 0

ls_returns = {}
for name in factor_names:
    ls_returns[name] = long_short_portfolio(factors[name], returns)

# robustness test 1 - top/bottom percentile cutoff
print("test 1: varying top/bottom percentile cutoff...")
pct_results = []
for pct in [0.1, 0.2, 0.3]:
    row = {"percentile": f"{int(pct*100)}%"}
    for name in factor_names:
        s = long_short_portfolio(factors[name], returns, top_pct=pct, bottom_pct=pct)
        row[name] = round(sharpe(s), 2)
    pct_results.append(row)
    print(f"  {int(pct*100)}%: {row}")

pd.DataFrame(pct_results).to_csv("results/robustness_percentile.csv", index=False)

# robustness test 2 - in sample vs out of sample
print("\ntest 2: in-sample vs out-of-sample (split 2019)...")
split = "2019-01-01"
for label, before in [("in-sample (2010-2018)", True), ("out-of-sample (2019-2025)", False)]:
    mask = common < split if before else common >= split
    print(f"\n  {label}")
    for name in factor_names:
        s = long_short_portfolio(factors[name].loc[mask], returns.loc[mask])
        print(f"    {name}: sharpe={sharpe(s):.2f}, return={s.mean()*252:.2%}")

# robustness test 3 - decade analysis
print("\ndecade analysis...")
for label, start, end in [
    ("2010-2014", "2010-01-01", "2014-12-31"),
    ("2015-2019", "2015-01-01", "2019-12-31"),
    ("2020-2025", "2020-01-01", "2025-12-31")
]:
    mask = (common >= start) & (common <= end)
    print(f"\n  {label}")
    for name in factor_names:
        s = long_short_portfolio(factors[name].loc[mask], returns.loc[mask])
        print(f"    {name}: sharpe={sharpe(s):.2f}")

# summary chart
fig, ax = plt.subplots(figsize=(12, 6))
periods = ["2010-2014", "2015-2019", "2020-2025"]
colors  = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
x       = np.arange(len(factor_names))
width   = 0.25

for i, (label, start, end) in enumerate([
    ("2010-2014", "2010-01-01", "2014-12-31"),
    ("2015-2019", "2015-01-01", "2019-12-31"),
    ("2020-2025", "2020-01-01", "2025-12-31")
]):
    mask    = (common >= start) & (common <= end)
    sharpes = []
    for name in factor_names:
        s = long_short_portfolio(factors[name].loc[mask], returns.loc[mask])
        sharpes.append(round(sharpe(s), 2))
    ax.bar(x + i * width, sharpes, width, label=label, alpha=0.7)

ax.set_xticks(x + width)
ax.set_xticklabels(factor_names, rotation=15)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("factor sharpe ratios by time period")
ax.set_ylabel("sharpe ratio")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/factor_decade_analysis.png", dpi=150, bbox_inches="tight")
plt.close()

print("\nsaved factor_decade_analysis.png")
print("done")

loading data...
test 1: varying top/bottom percentile cutoff...
  10%: {'percentile': '10%', 'momentum': np.float64(0.44), 'value': np.float64(-0.24), 'quality': np.float64(0.34), 'size': np.float64(-0.54), 'low_vol': np.float64(-0.2), 'profitability': np.float64(0.53)}
  20%: {'percentile': '20%', 'momentum': np.float64(0.4), 'value': np.float64(-0.24), 'quality': np.float64(0.37), 'size': np.float64(-0.46), 'low_vol': np.float64(-0.22), 'profitability': np.float64(0.53)}
  30%: {'percentile': '30%', 'momentum': np.float64(0.39), 'value': np.float64(-0.15), 'quality': np.float64(0.27), 'size': np.float64(-0.22), 'low_vol': np.float64(-0.35), 'profitability': np.float64(0.51)}

test 2: in-sample vs out-of-sample (split 2019)...

  in-sample (2010-2018)
    momentum: sharpe=0.10, return=1.34%
    value: sharpe=-0.12, return=-1.52%
    quality: sharpe=0.51, return=5.72%
    size: sharpe=-1.17, return=-9.73%
    low_vol: sharpe=-0.44, return=-6.15%
    profitability: sharpe=0.04, return=0

In [8]:
from google.colab import files
import os

for f in os.listdir("results"):
    files.download(f"results/{f}")
    print(f"downloaded: {f}")
print("done")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded: rolling_sharpe.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

downloaded: factor_cumulative_returns.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded: factor_correlation.png
done
